In [1]:
import sys
print("Python utilisé:", sys.executable)
print("Premiers chemins:", sys.path[:3])

import matplotlib
matplotlib.use('tkAgg')
from matplotlib import pyplot as plt
from PIL import Image
import torch
import numpy as np
import open3d as o3d
import requests
import dataclasses
from pathlib import Path
import depth_pro
import cv2 as cv
from depth_pro.depth_pro import create_model_and_transforms, DEFAULT_MONODEPTH_CONFIG_DICT

Python utilisé: c:\Users\mvm\open3d_vision\.venv\Scripts\python.exe
Premiers chemins: ['C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'C:\\Users\\mvm\\AppData\\Local\\Programs\\Python\\Python312\\Lib']
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:

img = cv.imread(r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg")
cv.imshow("Display window", img)
k = cv.waitKey(0) # Wait for a keystroke in the window

In [3]:
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt
 
img = cv.imread(r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg", cv.IMREAD_GRAYSCALE)
assert img is not None, "file could not be read, check with os.path.exists()"
img = cv.medianBlur(img,5)
 
ret,th1 = cv.threshold(img,127,255,cv.THRESH_BINARY)
th2 = cv.adaptiveThreshold(img,255,cv.ADAPTIVE_THRESH_MEAN_C,\
            cv.THRESH_BINARY,11,2)
th3 = cv.adaptiveThreshold(img,255,cv.ADAPTIVE_THRESH_GAUSSIAN_C,\
            cv.THRESH_BINARY,11,2)
 
titles = ['Original Image', 'Global Thresholding (v = 127)',
            'Adaptive Mean Thresholding', 'Adaptive Gaussian Thresholding']
images = [img, th1, th2, th3]
 
for i in range(4):
    plt.subplot(2,2,i+1),plt.imshow(images[i],'gray')
    plt.title(titles[i])
    plt.xticks([]),plt.yticks([])
plt.show()

In [4]:
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt
img = cv.imread(r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg", cv.IMREAD_GRAYSCALE)

assert img is not None, "file could not be read, check with os.path.exists()"

# global thresholding
ret1,th1 = cv.threshold(img,127,255,cv.THRESH_BINARY)

# Otsu's thresholding
ret2,th2 = cv.threshold(img,0,255,cv.THRESH_BINARY+cv.THRESH_OTSU)

# Otsu's thresholding after Gaussian filtering
blur = cv.GaussianBlur(img,(5,5),0)
ret3,th3 = cv.threshold(blur,0,255,cv.THRESH_BINARY+cv.THRESH_OTSU)

# plot all the images and their histograms
images = [img, 0, th1,
          img, 0, th2,
          blur, 0, th3]
titles = ['Original Noisy Image','Histogram','Global Thresholding (v=127)',
          'Original Noisy Image','Histogram',"Otsu's Thresholding",
          'Gaussian filtered Image','Histogram',"Otsu's Thresholding"]

for i in range(3):
    plt.subplot(3,3,i*3+1),plt.imshow(images[i*3],'gray')
    plt.title(titles[i*3]), plt.xticks([]), plt.yticks([])
    plt.subplot(3,3,i*3+2),plt.hist(images[i*3].ravel(),256)
    plt.title(titles[i*3+1]), plt.xticks([]), plt.yticks([])
    plt.subplot(3,3,i*3+3),plt.imshow(images[i*3+2],'gray')
    plt.title(titles[i*3+2]), plt.xticks([]), plt.yticks([])
plt.show()

In [5]:
img = cv.imread(r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg", cv.IMREAD_GRAYSCALE)
assert img is not None, "file could not be read, check with os.path.exists()"
blur = cv.GaussianBlur(img,(5,5),0)

# find normalized_histogram, and its cumulative distribution function
hist = cv.calcHist([blur],[0],None,[256],[0,256])
hist_norm = hist.ravel()/hist.sum()
Q = hist_norm.cumsum()

bins = np.arange(256)

fn_min = np.inf
thresh = -1

for i in range(1,256):
    p1,p2 = np.hsplit(hist_norm,[i]) # probabilities
    q1,q2 = Q[i],Q[255]-Q[i] # cum sum of classes
    if q1 < 1.e-6 or q2 < 1.e-6:
        continue
    b1,b2 = np.hsplit(bins,[i]) # weights

    # finding means and variances
    m1,m2 = np.sum(p1*b1)/q1, np.sum(p2*b2)/q2
    v1,v2 = np.sum(((b1-m1)**2)*p1)/q1,np.sum(((b2-m2)**2)*p2)/q2

    # calculates the minimization function
    fn = v1*q1 + v2*q2
    if fn < fn_min:
        fn_min = fn
        thresh = i

# find otsu's threshold value with OpenCV function
ret, otsu = cv.threshold(blur,0,255,cv.THRESH_BINARY+cv.THRESH_OTSU)
print( "{} {}".format(thresh,ret) )

145 144.0


In [6]:
# First, ensure you have a valid binary mask. Use 'otsu' from above (from cv.threshold).
# If you want to clean the image using the mask, use the binary mask
cleaned_img = cv.bitwise_and(img, img, mask=otsu)
cv.imshow("Display window", cleaned_img)
k = cv.waitKey(0) # Wait for a keystroke in the window

In [7]:
def revert_depth_image(depth_image):
    """
    Inverse la profondeur de l'image : proche <-> lointain.
    depth_image : array numpy 2D (H, W), valeurs de profondeur.
    Retourne une copie avec depth_inv = depth_max - depth + depth_min (range préservé, ordre inversé).
    """
    d = np.asarray(depth_image, dtype=np.float64)
    d_min, d_max = d.min(), d.max()
    return (d_max - d + d_min).astype(depth_image.dtype if hasattr(depth_image, 'dtype') else np.float32)

In [8]:
# 2) Modèle depth_pro + chargement / transform / overwrite PIL (ordre V2 strict)
CHECKPOINT = Path(r"C:\Users\mvm\open3d_vision\ml-depth-pro\checkpoints\depth_pro_alt.pt")
config = dataclasses.replace(DEFAULT_MONODEPTH_CONFIG_DICT, checkpoint_uri=str(CHECKPOINT))
model, transform = create_model_and_transforms(config=config)
model.eval()




DepthPro(
  (encoder): DepthProEncoder(
    (patch_encoder): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (mlp)

In [9]:
# 1) Carte de profondeur avec depth-pro sur l'image originale (pas de découpe avant)
from PIL import Image

PATH_IMG = r"C:\Users\mvm\open3d_vision\data\pile-of-soil-top-view-isolated-on-white-G1N4P8.jpg"
image_og, _, f_px = depth_pro.load_rgb(PATH_IMG)
prediction_1 = model.infer(transform(image_og), f_px=f_px)
depth_1 = prediction_1["depth"].squeeze().cpu().numpy()

In [10]:
def normalize_depth_map(depth):
    d_min = depth.min()
    d_max = depth.max()
    if d_max > d_min:
        return (depth - d_min) / (d_max - d_min)
    else:
        return np.zeros_like(depth)

depth_1 = normalize_depth_map(depth_1)

In [11]:
# 2) Filtre sur l'image RGB : chargement de l'image originale et création du masque th2 (Otsu)
img = cv.imread(PATH_IMG)
assert img is not None, "Image non trouvée : vérifier PATH_IMG"
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
img_gray = cv.GaussianBlur(img_gray, (5, 5), 0)
_, th2 = cv.threshold(img_gray, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

In [12]:
# 3) Découpe de la carte de profondeur avec le filtre th2 → image avec transparence (BGRA)
# Contenu = profondeur en colormap plasma ; alpha = th2 (opaque où th2 > 0, transparent ailleurs).

if th2.shape[:2] != depth_1.shape[:2]:
    th2_resized = cv.resize(th2, (depth_1.shape[1], depth_1.shape[0]), interpolation=cv.INTER_NEAREST)
else:
    th2_resized = th2
alpha = np.where(th2_resized > 0, 255, 0).astype(np.uint8)
depth_uint8 = (np.clip(depth_1, 0, 1) * 255).astype(np.uint8)
depth_bgr = cv.applyColorMap(depth_uint8, cv.COLORMAP_PLASMA)
depth_bgra = cv.merge([depth_bgr[:,:,0], depth_bgr[:,:,1], depth_bgr[:,:,2], alpha])
cv.imwrite("depth_decoupe_transparente.png", depth_bgra)
depth_bgra = revert_depth_image(depth_bgra)
plt.figure(figsize=(10, 8))
plt.imshow(cv.cvtColor(depth_bgra, cv.COLOR_BGRA2RGBA))
plt.axis("off")
plt.title("Carte de profondeur découpée avec transparence (alpha = filtre th2)")
plt.tight_layout()
plt.show()


In [13]:
# Modélisation de depth_bgra en un nuage de points
# z = vraie profondeur (depth_1), pas la couleur plasma ; x,y normalisés pour que le relief soit visible.

import open3d as o3d

h, w = depth_bgra.shape[:2]
assert depth_1.shape[:2] == (h, w), "depth_1 doit avoir la même taille (h,w) que depth_bgra"

# Grille des pixels (même ordre C que depth_1.flatten())
yy, xx = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
xx_flat = xx.ravel()
yy_flat = yy.ravel()

# Profondeur réelle (depth_1) pour l'altitude z, pas la couleur du colormap
z_flat = np.asarray(depth_1, dtype=np.float64).ravel()

# Couleurs depuis depth_bgra : BGR → RGB pour Open3D
colors = (depth_bgra[..., [2, 1, 0]].reshape(-1, 3).astype(np.float64) / 255.0)

# Masque alpha (même ordre que les tableaux aplatis)
alpha_flat = depth_bgra[..., 3].ravel() if depth_bgra.shape[2] == 4 else np.ones(h * w, dtype=np.uint8) * 255
mask = alpha_flat > 0

# x,y normalisés en [0,1] pour que le relief z (déjà en [0,1]) soit visible
x = xx_flat[mask].astype(np.float64) / (w - 1) if w > 1 else xx_flat[mask].astype(np.float64)
y = yy_flat[mask].astype(np.float64) / (h - 1) if h > 1 else yy_flat[mask].astype(np.float64)
z = z_flat[mask]
color = colors[mask]

points = np.stack([x, y, z], axis=1)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(color)

o3d.io.write_point_cloud("depth_bgra_pointcloud.ply", pcd)
o3d.visualization.draw_geometries([pcd], window_name="Nuage de points (z = carte de profondeur)")


In [14]:
# Reconstruction de surface par alpha shape sur le nuage de points (pcd) avec Open3D

import numpy as np

# L'alpha shape n'a pas besoin de normales
alpha = 0.1  # Ajuste ce paramètre pour changer la "finesse" du maillage

# Crée le maillage alpha shape
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd, alpha)

# Optionnel : supprime les triangles avec des arêtes trop longues (affine les bords du maillage)
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_triangles()
mesh.remove_duplicated_vertices()
mesh.remove_non_manifold_edges()

# Sauvegarder et visualiser la mesh
o3d.io.write_triangle_mesh("alpha_shape_mesh.ply", mesh)
o3d.visualization.draw_geometries([mesh], window_name="Maillage Alpha Shape du nuage de points")


In [15]:
import matplotlib.pyplot as plt
depth_1 = revert_depth_image(depth_1)
plt.imshow(depth_1, cmap='plasma')
plt.colorbar(label='Profondeur normalisée')
plt.title("Carte de profondeur générée")
plt.axis('off')
plt.show()